# BRAMASTRA K8: Two-T4 Cognition and Recursive Improvement Campaign

**Owner-launched.** Allocation: max **600 elapsed min / 1200 GPU-min** on 2x T4 (fits the 12-hour Kaggle GPU session cap with margin).

**Important:** kernel restarts do NOT reset the clock (SQLite ledger).
If E0 already ran, the full-run cell skips it.
Stop new training by minute 570. Hard stop before 600 (30-minute export reserve).

**Kaggle setup:** select accelerator `GPU T4 x2`, then attach two Kaggle Dataset inputs: (1) an exported BRAMASTRA source tree containing `pyproject.toml` and `bramastra_lab/`; (2) the prebuilt K8 data bundle containing `manifest.json`. The notebook discovers both inputs under `/kaggle/input`. No clone, pip install, Internet access, or local code edit is required in the GPU session.

For a new independent campaign, set `BRAMASTRA_RUN_ID` before running the setup cell. Leave it unchanged when resuming the same Kaggle session. Results are written under `/kaggle/working` and are retained when the notebook is saved as a Kaggle version.

In [ ]:
import importlib.util
import json
import os
import re
import subprocess
import sys
import uuid
from pathlib import Path

WORKING = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

def _children(path):
    if not path.is_dir():
        return []
    try:
        return sorted(path.iterdir(), key=lambda item: item.name)
    except OSError:
        return []

def _source_candidates():
    configured = os.environ.get('BRAMASTRA_REPO')
    if configured:
        yield Path(configured)
    yield Path.cwd()
    yield WORKING / 'An-Ra-the-new-AGI'
    for parent in (INPUT, WORKING):
        for child in _children(parent):
            yield child
            for grandchild in _children(child):
                yield grandchild

def _is_source_tree(path):
    return (path / 'pyproject.toml').is_file() and (path / 'bramastra_lab').is_dir()

REPO = next((path.resolve() for path in _source_candidates() if _is_source_tree(path)), None)
if REPO is None:
    raise RuntimeError(
        'BRAMASTRA source was not found. Attach a Kaggle Dataset containing '
        'pyproject.toml and bramastra_lab/, then rerun this cell. Optional override: BRAMASTRA_REPO.')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if importlib.util.find_spec('bramastra_lab') is None:
    raise RuntimeError(f'BRAMASTRA source exists at {REPO}, but Python cannot import it.')

REQUIRED_MODULES = ('torch', 'numpy', 'pytest')
missing_modules = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(
        'Kaggle runtime is missing required modules: ' + ', '.join(missing_modules) +
        '. Select a compatible Kaggle Python image or add the pinned dependency before starting this campaign.')

def run_k8(label, *arguments):
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(REPO) + os.pathsep + environment.get('PYTHONPATH', '')
    command = [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8', *arguments]
    result = subprocess.run(command, cwd=REPO, env=environment, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, file=sys.stderr)
        raise RuntimeError(f'{label} failed with exit code {result.returncode}')
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    return result

requested_run_id = os.environ.get('BRAMASTRA_RUN_ID')
run_id_file = WORKING / 'bramastra-k8-run-id.txt'
if requested_run_id:
    RUN_ID = requested_run_id
elif run_id_file.is_file():
    RUN_ID = run_id_file.read_text(encoding='utf-8').strip()
else:
    RUN_ID = f'k8-{uuid.uuid4().hex[:12]}'
    run_id_file.write_text(RUN_ID + '\n', encoding='utf-8')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]{2,79}', RUN_ID):
    raise RuntimeError('BRAMASTRA_RUN_ID must be 3-80 safe filename characters.')
INSTANCE_ID = uuid.uuid4().hex[:12]
RUN_ROOT = WORKING / 'bramastra-k8' / RUN_ID
RUN_DIR = RUN_ROOT / 'campaign'
BUILD_REPORT_DIR = RUN_ROOT / f'build-verification-{INSTANCE_ID}'
EXPORT_DIR = RUN_ROOT / f'export-{INSTANCE_ID}'

import torch
if tuple(int(part) for part in re.findall(r'\d+', torch.__version__)[:2]) < (2, 6):
    raise RuntimeError(f'K8 requires torch>=2.6; Kaggle provided torch {torch.__version__}.')
from bramastra_lab.research.runtime.provenance import source_identity
src = source_identity()
print('source:', json.dumps(src, indent=2))
print('source root:', REPO)
print('run ID:', RUN_ID)
print('GPU count:', torch.cuda.device_count())
if torch.cuda.device_count() != 2:
    raise RuntimeError('K8 requires exactly two visible GPUs. In Kaggle, select GPU T4 x2 and restart the session.')
for i in range(torch.cuda.device_count()):
    print(f'  cuda:{i}:', torch.cuda.get_device_name(i))
print('Phases: E0(0-38) E1(38-190) E2(190-247) E3(247-323) E4(323-399) E5(399-570) E6(570-600)')
from bramastra_lab.research.config import tokenizer_identity
from bramastra_lab.research.campaigns.phases.ops import k8_campaign_config
cfg = k8_campaign_config()
print('config:', cfg.identity())
print('tokenizer:', tokenizer_identity())
print('model: vocab260/L8/W256/H4/FFN704/ctx512 params6493952')
print('allocation: single 600min campaign / 1200 provisioned GPU-min; E0 and full share it')


## 2. Discover the Prebuilt Data Bundle (read-only)

The full bundle is an input artifact. This cell refuses a missing, malformed, or wrong-schema bundle before any verification or GPU allocation begins.

In [ ]:
def _is_k8_bundle(path):
    manifest = path / 'manifest.json'
    if not manifest.is_file():
        return False
    try:
        return json.loads(manifest.read_text(encoding='utf-8')).get('schema') == 'bramastra-k8-data/v1'
    except (OSError, json.JSONDecodeError):
        return False

def _bundle_candidates():
    configured = os.environ.get('BRAMASTRA_BUNDLE_DIR')
    if configured:
        yield Path(configured)
    for child in _children(INPUT):
        yield child
        for grandchild in _children(child):
            yield grandchild

BUNDLE_PATH = next((path.resolve() for path in _bundle_candidates() if _is_k8_bundle(path)), None)
if BUNDLE_PATH is None:
    raise RuntimeError(
        'K8 data bundle was not found. Attach the prebuilt dataset with a '
        'bramastra-k8-data/v1 manifest.json, then rerun. Optional override: BRAMASTRA_BUNDLE_DIR. '
        'Prepare the full 4096/256/256/128 bundle before this GPU session; do not generate it here.')
BUNDLE_DIR = str(BUNDLE_PATH)
print('using K8 bundle:', BUNDLE_DIR)


## 3. Validate Data

In [ ]:
run_k8('validate', 'validate', '--bundle', BUNDLE_DIR)


## 3b. Build Verification (pre-allocation, zero optimizer commits)

Runs the registered local contract checks and real no-step production interfaces, then writes an evidence-backed build report. The campaign must not start unless the build verifies.


In [ ]:
run_k8(
    'build verification', 'verify-build', '--data', BUNDLE_DIR,
    '--report-dir', str(BUILD_REPORT_DIR), '--no-updates',
    '--notebook', str(REPO / 'notebooks' / 'bramastra_k8.ipynb'))
print('build report:', BUILD_REPORT_DIR / 'build_verification.json')


## 4. E0 Gate

In [ ]:
run_k8(
    'E0 gate', 'run', '--mode', 'e0', '--run-dir', str(RUN_DIR),
    '--data', BUNDLE_DIR, '--max-wall-minutes', '600',
    '--devices', 'cuda:0,cuda:1', '--build-report', str(BUILD_REPORT_DIR),
    '--precision', 'fp16_autocast')


## 5. Full Campaign (E0 through E6)
If E0 already ran, this cell skips it and uses remaining time.

In [ ]:
run_k8(
    'full campaign', 'run', '--mode', 'full', '--run-dir', str(RUN_DIR),
    '--data', BUNDLE_DIR, '--max-wall-minutes', '600',
    '--devices', 'cuda:0,cuda:1', '--build-report', str(BUILD_REPORT_DIR),
    '--precision', 'fp16_autocast')


## 6. Summarize

In [ ]:
run_k8('summarize', 'summarize', '--run-dir', str(RUN_DIR))


## 7. Export

In [ ]:
run_k8('export', 'export', '--run-dir', str(RUN_DIR), '--out', str(EXPORT_DIR))
print('K8 export:', EXPORT_DIR)
print('Save a Kaggle version to retain this /kaggle/working output.')
